# Scraping de coches de segunda mano 

### Que queremos 
- hacer scraping de paginas web de coches de segunda mano
- comparar precios de paginas de coches de segunda mano 
- utilizar rag para guardar los precios de los coches


### login en el modelo 

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')

MODEL = "gemini-2.0-flash"
openai = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta", api_key=api_key)

response = openai.chat.completions.create(
 model=MODEL,
 messages=[{"role": "user", "content": "¿Cuánto son 2 + 2?"}]
)

print(response.choices[0].message.content)

2 + 2 = 4



In [57]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import pandas as pd
import time

# Opciones para Selenium (modo headless: no abre ventana)
options = Options()
options.add_argument('--headless')
options.add_argument('--window-size=1920,1080')

driver = webdriver.Chrome(options=options)
url = "https://www.autofesa.com/coches-segunda-mano"
driver.get(url)
time.sleep(3)  # Espera a que cargue la web


# Solo selecciona los vehicle-list__item de primer nivel (no anidados)
car_elements = driver.find_elements(By.CSS_SELECTOR, ".vehicle-list > .vehicle-list__item")

resultados = []
for idx, car in enumerate(car_elements, 1):
    try:
        title = car.find_element(By.CLASS_NAME, "vehicle-card__title").text.strip()
    except:
        title = "Sin título"
    try:
        price = car.find_element(By.CLASS_NAME, "vehicle-card__price").text.strip()
    except:
        price = "Sin precio"
    try:
        link = car.find_element(By.CSS_SELECTOR, "a").get_attribute("href")
    except:
        link = "Sin enlace"
    resultados.append({"#": idx, "Modelo": title, "Precio": price, "Link": link})

driver.quit()

In [58]:
# Mostrar resultados como tabla en el notebook
df = pd.DataFrame(resultados)
df

,#,Modelo,Precio,Link
0,1,Abarth 500 1.4T JET 595 165CV 70TH ANNIVERSARY...,17.350\n€\nOFERTA,https://www.autofesa.com/coches-de-ocasion/aba...
1,2,Aixam S10 SPORT S10 SPORT,Sin precio,https://www.autofesa.com/coches-de-ocasion/aix...
2,3,Aixam S8 COUPE S8 COUPE 9CV AUTO 3P # CUERO,10.450\n€\nOFERTA,https://www.autofesa.com/coches-de-ocasion/aix...
3,4,,,https://www.autofesa.com/coches-de-ocasion/alf...
4,5,,,https://www.autofesa.com/coches-de-ocasion/alf...
5,6,,,https://www.autofesa.com/coches-de-ocasion/alf...
6,7,,,https://www.autofesa.com/coches-de-ocasion/alf...
7,8,,,https://www.autofesa.com/coches-de-ocasion/alf...
8,9,,,https://www.autofesa.com/coches-de-ocasion/alf...
9,10,,,https://www.autofesa.com/coches-de-ocasion/alf...


### Diagnóstico del problema - ¿Por qué solo 3 coches?

In [59]:
# Vamos a diagnosticar el problema
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time

options = Options()
options.add_argument('--headless')
options.add_argument('--window-size=1920,1080')

driver = webdriver.Chrome(options=options)
url = "https://www.autofesa.com/coches-segunda-mano"
driver.get(url)

print("1. Esperando que cargue la página...")
time.sleep(5)  # Esperamos más tiempo

print("2. Verificando si hay elementos con clase 'vehicle-list'...")
vehicle_lists = driver.find_elements(By.CSS_SELECTOR, ".vehicle-list")
print(f"   - Contenedores .vehicle-list encontrados: {len(vehicle_lists)}")

print("3. Verificando elementos con selectores alternativos...")
# Probar diferentes selectores
selectores_alternativos = [
    ".vehicle-list__item",
    "[class*='vehicle']",
    "[class*='car']",
    "[class*='item']",
    ".card",
    ".listing"
]

for selector in selectores_alternativos:
    elementos = driver.find_elements(By.CSS_SELECTOR, selector)
    print(f"   - Selector '{selector}': {len(elementos)} elementos")

print("4. Verificando si la página necesita scroll...")
# Hacer scroll para cargar más contenido
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(3)

print("5. Verificando de nuevo después del scroll...")
car_elements_after_scroll = driver.find_elements(By.CSS_SELECTOR, ".vehicle-list > .vehicle-list__item")
print(f"   - Elementos después del scroll: {len(car_elements_after_scroll)}")

print("6. Verificando si hay botón 'Cargar más' o paginación...")
load_more_buttons = driver.find_elements(By.CSS_SELECTOR, "[class*='load'], [class*='more'], [class*='next']")
print(f"   - Botones 'cargar más' encontrados: {len(load_more_buttons)}")

print("7. Obteniendo el HTML de la página para inspección...")
page_source = driver.page_source
print(f"   - Tamaño del HTML: {len(page_source)} caracteres")

# Buscar patrones en el HTML
if "vehicle-list__item" in page_source:
    print("   - ✓ El HTML contiene 'vehicle-list__item'")
else:
    print("   - ✗ El HTML NO contiene 'vehicle-list__item'")

driver.quit()

1. Esperando que cargue la página...
2. Verificando si hay elementos con clase 'vehicle-list'...
   - Contenedores .vehicle-list encontrados: 1
3. Verificando elementos con selectores alternativos...
   - Selector '.vehicle-list__item': 30 elementos
   - Selector '[class*='vehicle']': 399 elementos
   - Selector '[class*='car']': 356 elementos
   - Selector '[class*='item']': 363 elementos
   - Selector '.card': 0 elementos
   - Selector '.listing': 0 elementos
4. Verificando si la página necesita scroll...
5. Verificando de nuevo después del scroll...
   - Elementos después del scroll: 30
6. Verificando si hay botón 'Cargar más' o paginación...
   - Botones 'cargar más' encontrados: 911
7. Obteniendo el HTML de la página para inspección...
   - Tamaño del HTML: 968293 caracteres
   - ✓ El HTML contiene 'vehicle-list__item'


### Versión mejorada del scraper

In [60]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

def scrape_autofesa_mejorado():
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    
    driver = webdriver.Chrome(options=options)
    resultados = []
    
    try:
        url = "https://www.autofesa.com/coches-segunda-mano"
        print(f"Accediendo a: {url}")
        driver.get(url)
        
        # Esperar a que la página cargue completamente
        wait = WebDriverWait(driver, 10)
        print("Esperando a que cargue el contenido...")
        
        # Intentar múltiples estrategias de espera
        try:
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".vehicle-list__item")))
            print("✓ Elementos vehicle-list__item encontrados")
        except:
            print("⚠ No se encontraron elementos vehicle-list__item, probando otros selectores...")
        
        # Hacer scroll para cargar contenido dinámico
        print("Haciendo scroll para cargar más contenido...")
        for i in range(3):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
        
        # Probar múltiples selectores
        selectores = [
            ".vehicle-list > .vehicle-list__item",
            ".vehicle-list__item",
            "[class*='vehicle-card']",
            "[class*='car-item']",
            ".card",
            "[data-testid*='vehicle']"
        ]
        
        car_elements = []
        for selector in selectores:
            elements = driver.find_elements(By.CSS_SELECTOR, selector)
            if elements:
                print(f"✓ Encontrados {len(elements)} elementos con selector: {selector}")
                car_elements = elements
                break
            else:
                print(f"✗ 0 elementos con selector: {selector}")
        
        if not car_elements:
            print("No se encontraron elementos de coches. Probando selectores más genéricos...")
            # Último intento con selectores muy genéricos
            generic_selectors = ["div[class*='vehicle']", "div[class*='car']", "article", ".item"]
            for selector in generic_selectors:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                if len(elements) > 5:  # Solo si encontramos un número razonable
                    print(f"✓ Usando selector genérico: {selector} ({len(elements)} elementos)")
                    car_elements = elements[:20]  # Limitar a 20 para evitar elementos irrelevantes
                    break
        
        print(f"Procesando {len(car_elements)} elementos...")
        
        for idx, car in enumerate(car_elements, 1):
            try:
                # Múltiples selectores para el título
                title_selectors = [
                    ".vehicle-card__title",
                    ".card-title",
                    ".title",
                    "h2", "h3", "h4",
                    "[class*='title']",
                    "[class*='name']"
                ]
                
                title = "Sin título"
                for sel in title_selectors:
                    try:
                        title_elem = car.find_element(By.CSS_SELECTOR, sel)
                        title = title_elem.text.strip()
                        if title:
                            break
                    except:
                        continue
                
                # Múltiples selectores para el precio
                price_selectors = [
                    ".vehicle-card__price",
                    ".price",
                    "[class*='price']",
                    "[class*='cost']"
                ]
                
                price = "Sin precio"
                for sel in price_selectors:
                    try:
                        price_elem = car.find_element(By.CSS_SELECTOR, sel)
                        price = price_elem.text.strip()
                        if price:
                            break
                    except:
                        continue
                
                # Link
                try:
                    link = car.find_element(By.CSS_SELECTOR, "a").get_attribute("href")
                except:
                    link = "Sin enlace"
                
                # Solo agregar si tiene título válido
                if title and title != "Sin título" and len(title) > 3:
                    resultados.append({
                        "#": idx,
                        "Modelo": title,
                        "Precio": price,
                        "Link": link
                    })
                    
            except Exception as e:
                print(f"Error procesando elemento {idx}: {e}")
                continue
    
    except Exception as e:
        print(f"Error general: {e}")
    
    finally:
        driver.quit()
    
    return resultados

# Ejecutar el scraper mejorado
print("Iniciando scraping mejorado...")
resultados_mejorados = scrape_autofesa_mejorado()
print(f"\\n✓ Scraping completado. Se encontraron {len(resultados_mejorados)} coches.")

Iniciando scraping mejorado...
Accediendo a: https://www.autofesa.com/coches-segunda-mano
Esperando a que cargue el contenido...
✓ Elementos vehicle-list__item encontrados
Haciendo scroll para cargar más contenido...
✓ Encontrados 30 elementos con selector: .vehicle-list > .vehicle-list__item
Procesando 30 elementos...
\n✓ Scraping completado. Se encontraron 0 coches.


In [61]:
# Mostrar resultados mejorados
if resultados_mejorados:
    df_mejorado = pd.DataFrame(resultados_mejorados)
    print(f"Total de coches encontrados: {len(df_mejorado)}")
    print("\\nPrimeros 10 resultados:")
    display(df_mejorado.head(10))
    
    # Guardar en CSV
    df_mejorado.to_csv('autofesa_coches_mejorado.csv', index=False)
    print("\\n✓ Datos guardados en 'autofesa_coches_mejorado.csv'")
else:
    print("❌ No se encontraron coches. Posibles causas:")
    print("1. La página ha cambiado su estructura")
    print("2. Hay medidas anti-bot")
    print("3. La página carga contenido dinámicamente con JavaScript complejo")
    print("4. Se requiere interacción del usuario (cookies, captcha, etc.)")

❌ No se encontraron coches. Posibles causas:
1. La página ha cambiado su estructura
2. Hay medidas anti-bot
3. La página carga contenido dinámicamente con JavaScript complejo
4. Se requiere interacción del usuario (cookies, captcha, etc.)


### Inspección detallada de la estructura HTML

In [62]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time

def inspeccionar_estructura_detallada():
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--window-size=1920,1080')
    
    driver = webdriver.Chrome(options=options)
    
    try:
        url = "https://www.autofesa.com/coches-segunda-mano"
        driver.get(url)
        time.sleep(5)
        
        # Encontrar los primeros 3 elementos de coches
        car_elements = driver.find_elements(By.CSS_SELECTOR, ".vehicle-list__item")[:3]
        
        print(f"Inspeccionando los primeros {len(car_elements)} coches:")
        print("=" * 60)
        
        for i, car in enumerate(car_elements, 1):
            print(f"\\n🚗 COCHE {i}:")
            print("-" * 30)
            
            # Obtener todo el HTML del elemento
            html_content = car.get_attribute('outerHTML')
            
            # Buscar todos los elementos de texto dentro del coche
            all_text_elements = car.find_elements(By.CSS_SELECTOR, "*")
            
            print("📝 Textos encontrados:")
            for elem in all_text_elements:
                text = elem.text.strip()
                if text and len(text) > 2:  # Solo textos significativos
                    tag_name = elem.tag_name
                    class_name = elem.get_attribute('class') or 'sin-clase'
                    print(f"   [{tag_name}.{class_name}]: {text[:50]}...")
            
            # Buscar específicamente elementos que podrían ser título
            print("\\n🏷️ Posibles títulos:")
            title_selectors = ["h1", "h2", "h3", "h4", "h5", ".title", "[class*='title']", "[class*='name']"]
            for selector in title_selectors:
                try:
                    elements = car.find_elements(By.CSS_SELECTOR, selector)
                    for elem in elements:
                        text = elem.text.strip()
                        if text:
                            print(f"   {selector}: {text}")
                except:
                    pass
            
            # Buscar específicamente elementos que podrían ser precio
            print("\\n💰 Posibles precios:")
            price_selectors = [".price", "[class*='price']", "[class*='cost']", "[class*='euro']", "[class*='€']"]
            for selector in price_selectors:
                try:
                    elements = car.find_elements(By.CSS_SELECTOR, selector)
                    for elem in elements:
                        text = elem.text.strip()
                        if text:
                            print(f"   {selector}: {text}")
                except:
                    pass
            
            # Mostrar una muestra del HTML (primeros 300 caracteres)
            print(f"\\n📄 HTML (muestra): {html_content[:300]}...")
            print("\\n" + "="*60)
    
    except Exception as e:
        print(f"Error: {e}")
    
    finally:
        driver.quit()

# Ejecutar inspección detallada
inspeccionar_estructura_detallada()

Inspeccionando los primeros 3 coches:
\n🚗 COCHE 1:
------------------------------
📝 Textos encontrados:
   [div.vehicle-card]: Dirección asistida, Bluetooth, Sensor de lluvia, C...
   [div.vehicle-card__equipment]: Dirección asistida, Bluetooth, Sensor de lluvia, C...
   [div.text collapse]: Dirección asistida, Bluetooth, Sensor de lluvia, C...
   [div.vehicle-card__title]: Abarth 500 1.4T JET 595 165CV 70TH ANNIVERSARY 3P ...
   [a.sin-clase]: Abarth 500 1.4T JET 595 165CV 70TH ANNIVERSARY 3P ...
   [span.make]: Abarth...
   [span.model]: 500...
   [span.version]: 1.4T JET 595 165CV 70TH ANNIVERSARY 3P # IVA DEDUC...
   [div.vehicle-card__content]: 2020 83.900km Gasolina Manual Utilitario
17.350
€
...
   [div.vehicle-card__features]: 2020 83.900km Gasolina Manual Utilitario...
   [ul.list]: 2020 83.900km Gasolina Manual Utilitario...
   [li.item]: 2020...
   [li.item]: 83.900km...
   [li.item]: Gasolina...
   [li.item]: Manual...
   [li.item]: Utilitario...
   [div.vehicle-card__price

### 🎯 Scraper final corregido

In [63]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

def scrape_autofesa_final():
    """Scraper final basado en la inspección detallada"""
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    
    driver = webdriver.Chrome(options=options)
    resultados = []
    
    try:
        url = "https://www.autofesa.com/coches-segunda-mano"
        print(f"🔗 Accediendo a: {url}")
        driver.get(url)
        
        # Esperar a que cargue el contenido
        wait = WebDriverWait(driver, 15)
        print("⏳ Esperando que cargue la página...")
        
        # Esperar a que aparezcan los coches
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".vehicle-list__item")))
        time.sleep(3)  # Tiempo adicional para asegurar carga completa
        
        # Encontrar todos los coches
        car_elements = driver.find_elements(By.CSS_SELECTOR, ".vehicle-list__item")
        print(f"🚗 Encontrados {len(car_elements)} coches")
        
        for idx, car in enumerate(car_elements, 1):
            try:
                # Extraer título usando el selector correcto
                try:
                    title_elem = car.find_element(By.CSS_SELECTOR, ".vehicle-card__title")
                    title = title_elem.text.strip()
                except:
                    title = "Sin título"
                
                # Extraer precio usando el selector correcto
                try:
                    price_elem = car.find_element(By.CSS_SELECTOR, ".vehicle-card__price")
                    price = price_elem.text.strip()
                    if not price:
                        price = "Precio no disponible"
                except:
                    price = "Precio no disponible"
                
                # Extraer link
                try:
                    link_elem = car.find_element(By.CSS_SELECTOR, "a")
                    link = link_elem.get_attribute("href")
                except:
                    link = "Sin enlace"
                
                # Extraer información adicional (año, km, combustible)
                try:
                    features = car.find_elements(By.CSS_SELECTOR, ".vehicle-card__features .list .item")
                    additional_info = " | ".join([f.text.strip() for f in features if f.text.strip()])
                except:
                    additional_info = "Sin información adicional"
                
                # Agregar al resultado
                resultados.append({
                    "#": idx,
                    "Modelo": title,
                    "Precio": price,
                    "Información": additional_info,
                    "Link": link
                })
                
                if idx <= 5:  # Mostrar progreso para los primeros 5
                    print(f"   ✓ Coche {idx}: {title[:30]}... - {price}")
                    
            except Exception as e:
                print(f"   ❌ Error procesando coche {idx}: {e}")
                continue
    
    except Exception as e:
        print(f"❌ Error general: {e}")
    
    finally:
        driver.quit()
    
    return resultados

# Ejecutar el scraper final
print("🚀 Iniciando scraper final corregido...")
resultados_final = scrape_autofesa_final()
print(f"\\n✅ Scraping completado. Total: {len(resultados_final)} coches extraídos.")

🚀 Iniciando scraper final corregido...
🔗 Accediendo a: https://www.autofesa.com/coches-segunda-mano
⏳ Esperando que cargue la página...
🚗 Encontrados 30 coches
   ✓ Coche 1: Abarth 500 1.4T JET 595 165CV ... - 17.350
€
OFERTA
   ✓ Coche 2: Aixam S10 SPORT S10 SPORT... - Precio no disponible
   ✓ Coche 3: Aixam S8 COUPE S8 COUPE 9CV AU... - 10.450
€
OFERTA
   ✓ Coche 4: ... - Precio no disponible
   ✓ Coche 5: ... - Precio no disponible
\n✅ Scraping completado. Total: 30 coches extraídos.


In [69]:
# Mostrar y analizar resultados finales
if resultados_final:
    df_final = pd.DataFrame(resultados_final)
    
    print(f"📊 RESUMEN DE RESULTADOS:")
    print(f"   • Total de coches: {len(df_final)}")
    print(f"   • Coches con precio: {len(df_final[df_final['Precio'] != 'Precio no disponible'])}")
    print(f"   • Coches sin precio: {len(df_final[df_final['Precio'] == 'Precio no disponible'])}")
    
    print("\\n🚗 PRIMEROS 10 COCHES:")
    display(df_final.head(10))
    
    # Guardar en CSV
    filename = 'autofesa_coches_final.csv'
    df_final.to_csv(filename, index=False, encoding='utf-8')
    print(f"\\n💾 Datos guardados en '{filename}'")
    
    # Mostrar algunos ejemplos de precios
    coches_con_precio = df_final[df_final['Precio'] != 'Precio no disponible']
    if not coches_con_precio.empty:
        print("\\n💰 EJEMPLOS DE PRECIOS:")
        for i, row in coches_con_precio.head(5).iterrows():
            print(f"   • {row['Modelo'][:40]}... → {row['Precio']}")
    
else:
    print("❌ No se pudieron extraer datos. Revisa la conexión o la estructura de la página.")

📊 RESUMEN DE RESULTADOS:
   • Total de coches: 30
   • Coches con precio: 0
   • Coches sin precio: 30
\n🚗 PRIMEROS 10 COCHES:


,#,Modelo,Precio,Información,Link
0,1,,Precio no disponible,,https://www.autofesa.com/coches-de-ocasion/aba...
1,2,,Precio no disponible,,https://www.autofesa.com/coches-de-ocasion/aix...
2,3,,Precio no disponible,,https://www.autofesa.com/coches-de-ocasion/aix...
3,4,,Precio no disponible,,https://www.autofesa.com/coches-de-ocasion/alf...
4,5,,Precio no disponible,,https://www.autofesa.com/coches-de-ocasion/alf...
5,6,,Precio no disponible,,https://www.autofesa.com/coches-de-ocasion/alf...
6,7,,Precio no disponible,,https://www.autofesa.com/coches-de-ocasion/alf...
7,8,,Precio no disponible,,https://www.autofesa.com/coches-de-ocasion/alf...
8,9,,Precio no disponible,,https://www.autofesa.com/coches-de-ocasion/alf...
9,10,,Precio no disponible,,https://www.autofesa.com/coches-de-ocasion/alf...


\n💾 Datos guardados en 'autofesa_coches_final.csv'


In [71]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

def scrape_autofesa_final():
    """Scraper mejorado para extraer todos los coches y precios aunque cambien las clases"""
    options = Options()
    # options.add_argument('--headless')  # Temporalmente desactivado para asegurar carga
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    
    driver = webdriver.Chrome(options=options)
    resultados = []
    
    try:
        url = "https://www.autofesa.com/coches-segunda-mano"
        print(f"🔗 Accediendo a: {url}")
        driver.get(url)
        
        wait = WebDriverWait(driver, 15)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".vehicle-list__item")))
        time.sleep(2)

        # ==== SCROLL INFINITO ====
        SCROLL_PAUSE_TIME = 2
        last_height = driver.execute_script("return document.body.scrollHeight")

        while True:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(SCROLL_PAUSE_TIME)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height

        car_elements = driver.find_elements(By.CSS_SELECTOR, ".vehicle-list__item")
        print(f"🚗 Encontrados {len(car_elements)} coches después del scroll")
        
        for idx, car in enumerate(car_elements, 1):
            try:
                # Título
                try:
                    title_elem = car.find_element(By.CSS_SELECTOR, ".vehicle-card__title")
                    title = title_elem.text.strip()
                except:
                    title = "Sin título"
                
                # Precio (varios selectores posibles)
                price = "Precio no disponible"
                price_selectors = [
                    ".vehicle-card__price",  # selector normal
                    ".vehicle-card__price--soldout",  # coches sin stock o vendidos
                    ".vehicle-card__price span"  # fallback a span
                ]
                for sel in price_selectors:
                    try:
                        price_elem = car.find_element(By.CSS_SELECTOR, sel)
                        if price_elem.text.strip():
                            price = price_elem.text.strip()
                            break
                    except:
                        continue
                
                # Link
                try:
                    link_elem = car.find_element(By.CSS_SELECTOR, "a")
                    link = link_elem.get_attribute("href")
                except:
                    link = "Sin enlace"
                
                # Información adicional
                try:
                    features = car.find_elements(By.CSS_SELECTOR, ".vehicle-card__features .list .item")
                    additional_info = " | ".join([f.text.strip() for f in features if f.text.strip()])
                except:
                    additional_info = "Sin información adicional"
                
                resultados.append({
                    "#": idx,
                    "Modelo": title,
                    "Precio": price,
                    "Información": additional_info,
                    "Link": link
                })
                
                if idx <= 5:
                    print(f"   ✓ Coche {idx}: {title[:30]}... - {price}")
                    
            except Exception as e:
                print(f"   ❌ Error procesando coche {idx}: {e}")
                continue
    
    except Exception as e:
        print(f"❌ Error general: {e}")
    
    finally:
        driver.quit()
    
    return resultados

# Ejecutar scraper
print("🚀 Iniciando scraper mejorado...")
resultados_final = scrape_autofesa_final()
print(f"\n✅ Scraping completado. Total: {len(resultados_final)} coches extraídos.")


🚀 Iniciando scraper mejorado...
🔗 Accediendo a: https://www.autofesa.com/coches-segunda-mano
🔗 Accediendo a: https://www.autofesa.com/coches-segunda-mano
🚗 Encontrados 30 coches después del scroll
🚗 Encontrados 30 coches después del scroll
   ✓ Coche 1: Abarth 500 1.4T JET 595 165CV ... - 17.350
€
OFERTA
   ✓ Coche 1: Abarth 500 1.4T JET 595 165CV ... - 17.350
€
OFERTA
   ✓ Coche 2: Aixam S10 SPORT S10 SPORT... - Precio no disponible
   ✓ Coche 2: Aixam S10 SPORT S10 SPORT... - Precio no disponible
   ✓ Coche 3: Aixam S8 COUPE S8 COUPE 9CV AU... - 10.450
€
OFERTA
   ✓ Coche 3: Aixam S8 COUPE S8 COUPE 9CV AU... - 10.450
€
OFERTA
   ✓ Coche 4: Alfa Romeo Giulietta 2.0 JTDM ... - 10.850
€
OFERTA
   ✓ Coche 4: Alfa Romeo Giulietta 2.0 JTDM ... - 10.850
€
OFERTA
   ✓ Coche 5: Alfa Romeo Giulietta GIULIETTA... - 16.850
€
OFERTA
   ✓ Coche 5: Alfa Romeo Giulietta GIULIETTA... - 16.850
€
OFERTA

✅ Scraping completado. Total: 30 coches extraídos.

✅ Scraping completado. Total: 30 coches extraíd

In [72]:
import pandas as pd

# Convertir lista de diccionarios a DataFrame
df = pd.DataFrame(resultados_final)

# Mostrar toda la tabla en consola
pd.set_option('display.max_rows', None)   # Mostrar todas las filas
pd.set_option('display.max_columns', None)  # Mostrar todas las columnas
pd.set_option('display.width', 1000)  # Evitar que se corte horizontalmente

print(df)


     #                                             Modelo                       Precio                                        Información                                               Link
0    1  Abarth 500 1.4T JET 595 165CV 70TH ANNIVERSARY...            17.350\n€\nOFERTA  2020 | 83.900km | Gasolina | Manual | Utilitar...  https://www.autofesa.com/coches-de-ocasion/aba...
1    2                          Aixam S10 SPORT S10 SPORT         Precio no disponible  2024 | 3.274km | Diesel | Manual | Utilitario ...  https://www.autofesa.com/coches-de-ocasion/aix...
2    3        Aixam S8 COUPE S8 COUPE 9CV AUTO 3P # CUERO            10.450\n€\nOFERTA  2015 | 44.144km | Diesel | Automático | Utilit...  https://www.autofesa.com/coches-de-ocasion/aix...
3    4  Alfa Romeo Giulietta 2.0 JTDM 140CV 5P # CUERO...            10.850\n€\nOFERTA  2013 | 131.800km | Diesel | Manual | Utilitari...  https://www.autofesa.com/coches-de-ocasion/alf...
4    5  Alfa Romeo Giulietta GIULIETTA 1.4T 170CV LUSS.

In [73]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

def scrape_autofesa_final():
    """Scraper mejorado para extraer todos los coches y precios aunque cambien las clases"""
    options = Options()
    # options.add_argument('--headless')  # Temporalmente desactivado para asegurar carga
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    
    driver = webdriver.Chrome(options=options)
    resultados = []
    
    try:
        url = "https://www.coches.net/segunda-mano/madrid/"
        print(f"🔗 Accediendo a: {url}")
        driver.get(url)
        
        wait = WebDriverWait(driver, 15)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".mt-ListAds")))
        time.sleep(2)

        # ==== SCROLL INFINITO ====
        SCROLL_PAUSE_TIME = 2
        last_height = driver.execute_script("return document.body.scrollHeight")

        while True:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(SCROLL_PAUSE_TIME)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height

        car_elements = driver.find_elements(By.CSS_SELECTOR, ".mt-ListAds-item")
        print(f"🚗 Encontrados {len(car_elements)} coches después del scroll")
        
        for idx, car in enumerate(car_elements, 1):
            try:
                # Título
                try:
                    title_elem = car.find_element(By.CSS_SELECTOR, ".card-ad-title")
                    title = title_elem.text.strip()
                except:
                    title = "Sin título"
                
                # Precio (varios selectores posibles)
                price = "Precio no disponible"
                price_selectors = [
                    ".card-adPrice-price",  # selector normal
                    ".vehicle-card__price--soldout",  # coches sin stock o vendidos
                    ".vehicle-card__price span"  # fallback a span
                ]
                for sel in price_selectors:
                    try:
                        price_elem = car.find_element(By.CSS_SELECTOR, sel)
                        if price_elem.text.strip():
                            price = price_elem.text.strip()
                            break
                    except:
                        continue
                
                # Link
                try:
                    link_elem = car.find_element(By.CSS_SELECTOR, "a")
                    link = link_elem.get_attribute("href")
                except:
                    link = "Sin enlace"
                
                # Información adicional
                try:
                    features = car.find_elements(By.CSS_SELECTOR, ".vehicle-card__features .list .item")
                    additional_info = " | ".join([f.text.strip() for f in features if f.text.strip()])
                except:
                    additional_info = "Sin información adicional"
                
                resultados.append({
                    "#": idx,
                    "Modelo": title,
                    "Precio": price,
                    "Información": additional_info,
                    "Link": link
                })
                
                if idx <= 5:
                    print(f"   ✓ Coche {idx}: {title[:30]}... - {price}")
                    
            except Exception as e:
                print(f"   ❌ Error procesando coche {idx}: {e}")
                continue
    
    except Exception as e:
        print(f"❌ Error general: {e}")
    
    finally:
        driver.quit()
    
    return resultados

# Ejecutar scraper
print("🚀 Iniciando scraper mejorado...")
resultados_final = scrape_autofesa_final()
print(f"\n✅ Scraping completado. Total: {len(resultados_final)} coches extraídos.")


🚀 Iniciando scraper mejorado...
🔗 Accediendo a: https://www.coches.net/segunda-mano/madrid/
🔗 Accediendo a: https://www.coches.net/segunda-mano/madrid/
🚗 Encontrados 40 coches después del scroll
   ✓ Coche 1: Sin título... - Precio no disponible
   ✓ Coche 2: Sin título... - Precio no disponible
   ✓ Coche 3: Sin título... - Precio no disponible
🚗 Encontrados 40 coches después del scroll
   ✓ Coche 1: Sin título... - Precio no disponible
   ✓ Coche 2: Sin título... - Precio no disponible
   ✓ Coche 3: Sin título... - Precio no disponible
   ✓ Coche 4: Sin título... - Precio no disponible
   ✓ Coche 5: Sin título... - Precio no disponible
   ✓ Coche 4: Sin título... - Precio no disponible
   ✓ Coche 5: Sin título... - Precio no disponible

✅ Scraping completado. Total: 40 coches extraídos.

✅ Scraping completado. Total: 40 coches extraídos.


In [74]:
import pandas as pd

# Convertir lista de diccionarios a DataFrame
df = pd.DataFrame(resultados_final)

# Mostrar toda la tabla en consola
pd.set_option('display.max_rows', None)   # Mostrar todas las filas
pd.set_option('display.max_columns', None)  # Mostrar todas las columnas
pd.set_option('display.width', 1000)  # Evitar que se corte horizontalmente

print(df)


     #      Modelo                Precio Información                                               Link
0    1  Sin título  Precio no disponible              https://www.coches.net/bmw-serie-4-420d-xdrive...
1    2  Sin título  Precio no disponible              https://www.coches.net/bmw-serie-4-420d-xdrive...
2    3  Sin título  Precio no disponible              https://www.coches.net/renault-megane-business...
3    4  Sin título  Precio no disponible              https://www.coches.net/renault-megane-business...
4    5  Sin título  Precio no disponible              https://www.coches.net/mercedes-benz-clase-cla...
5    6  Sin título  Precio no disponible              https://www.coches.net/mercedes-benz-clase-cla...
6    7  Sin título  Precio no disponible                                                     Sin enlace
7    8  Sin título  Precio no disponible                                                     Sin enlace
8    9  Sin título  Precio no disponible                        